<a href="https://colab.research.google.com/github/a7mdayman2002/Aitronix/blob/AI/MT5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate rouge-score fastapi pyngrok uvicorn nest-asyncio

In [ ]:
import os
import glob
import warnings
import threading
from pathlib import Path
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
from transformers import (pipeline,AutoTokenizer,AutoModelForSeq2SeqLM,
                          Seq2SeqTrainingArguments,Seq2SeqTrainer)
from datasets import Dataset
import evaluate

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Configure device
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"✓ Using device: {'GPU (CUDA)' if DEVICE == 0 else 'CPU'}")

✓ Using device: GPU (CUDA)


In [ ]:
from google.colab import drive
import zipfile
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Define paths
ZIP_PATH = "/content/drive/MyDrive/Colab Notebooks/MT5/archive (2).zip"
EXTRACT_TO = "/content/drive/MyDrive/Colab Notebooks/MT5/"

# Unzip the dataset
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_TO)

# Define article and summary paths
bbc_path = os.path.join(EXTRACT_TO, 'BBC News Summary')
ARTICLES_PATH = os.path.join(bbc_path, 'News Articles')
SUMMARIES_PATH = os.path.join(bbc_path, 'Summaries')

print("✓ Dataset ready")

Mounted at /content/drive
✓ Dataset ready


In [ ]:
def load_bbc_dataset(articles_path: str, summaries_path: str) -> pd.DataFrame:
    """
    Load BBC News dataset from disk.

    Args:
        articles_path: Path to articles directory
        summaries_path: Path to summaries directory

    Returns:
        DataFrame with 'article' and 'summary' columns
    """
    articles_dict = {}
    summaries_dict = {}

    # Load articles
    for txt_file in glob.glob(os.path.join(articles_path, '**/*.txt'), recursive=True):
        file_id = os.path.basename(txt_file)
        try:
            with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
                articles_dict[file_id] = f.read().strip()
        except Exception as e:
            print(f"Warning: Failed to read {txt_file}: {e}")

    # Load summaries
    for txt_file in glob.glob(os.path.join(summaries_path, '**/*.txt'), recursive=True):
        file_id = os.path.basename(txt_file)
        try:
            with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
                summaries_dict[file_id] = f.read().strip()
        except Exception as e:
            print(f"Warning: Failed to read {txt_file}: {e}")

    # Match articles with summaries
    data = []
    for file_id in articles_dict:
        if file_id in summaries_dict:
            data.append({
                'article': articles_dict[file_id],
                'summary': summaries_dict[file_id]
            })

    df = pd.DataFrame(data)
    print(f"✓ Loaded {len(df)} article-summary pairs")

    return df

# Load dataset
df = load_bbc_dataset(ARTICLES_PATH, SUMMARIES_PATH)



✓ Loaded 511 article-summary pairs


In [ ]:
# Initialize baseline T5-small model
baseline_summarizer = pipeline(
    "summarization",
    model="t5-small",
    device=DEVICE
)

print("✓ Baseline model loaded")

# Evaluate baseline on sample
sample_df = df.sample(n=5, random_state=42).reset_index(drop=True)

baseline_summaries = []
for article in tqdm(sample_df['article'], desc="Baseline evaluation"):
    summary = baseline_summarizer(
        article,
        max_length=130,
        min_length=30,
        do_sample=False
    )
    baseline_summaries.append(summary[0]['summary_text'])

sample_df['baseline_summary'] = baseline_summaries

# Calculate baseline ROUGE scores
rouge = evaluate.load('rouge')
baseline_results = rouge.compute(
    predictions=baseline_summaries,
    references=sample_df['summary'].tolist()
)

print("\n" + "="*50)
print("BASELINE MODEL PERFORMANCE")
print("="*50)
print(f"ROUGE-1: {baseline_results['rouge1']:.4f}")
print(f"ROUGE-2: {baseline_results['rouge2']:.4f}")
print(f"ROUGE-L: {baseline_results['rougeL']:.4f}")
print("="*50)



Device set to use cuda:0


✓ Baseline model loaded


Baseline evaluation:   0%|          | 0/5 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (896 > 512). Running this sequence through the model will result in indexing errors
Both `max_new_tokens` (=256) and `max_length`(=130) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Baseline evaluation: 100%|██████████| 5/5 [00:05<00:00,  1.14s/it]



BASELINE MODEL PERFORMANCE
ROUGE-1: 0.2944
ROUGE-2: 0.1824
ROUGE-L: 0.1992


In [ ]:
# Configuration
MODEL_NAME = "t5-small"
TRAIN_SIZE = 500
OUTPUT_DIR = "./fine_tuned_t5"

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Prepare training dataset
train_df = df.sample(n=min(TRAIN_SIZE, len(df)), random_state=42).reset_index(drop=True)
train_dataset = Dataset.from_pandas(train_df[['article', 'summary']])

print(f"✓ Training on {len(train_df)} samples")



✓ Training on 500 samples


In [ ]:
def preprocess_function(examples):
    """Tokenize articles and summaries for T5."""
    inputs = ["summarize: " + doc for doc in examples['article']]
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples['summary'],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# Tokenize dataset
tokenized_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=['article', 'summary'],
    desc="Tokenizing"
)

print("✓ Dataset tokenized")



Tokenizing:   0%|          | 0/500 [00:00<?, ? examples/s]

✓ Dataset tokenized


In [ ]:
# Training configuration
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"  # Disable wandb logging
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
)

print("✓ Trainer initialized")
print("\nStarting training (estimated 5-10 minutes)...\n")

# Train model
trainer.train()

# Save model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✓ Fine-tuning complete! Model saved to {OUTPUT_DIR}")

✓ Trainer initialized

Starting training (estimated 5-10 minutes)...



Step,Training Loss
50,2.559600
100,1.911200
150,1.555800
200,1.494800
250,1.228100
300,1.240500
350,1.311600



✓ Fine-tuning complete! Model saved to ./fine_tuned_t5


In [ ]:
# Load fine-tuned model
finetuned_summarizer = pipeline(
    "summarization",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=DEVICE
)

print("✓ Fine-tuned model loaded")

# Test on unseen samples
test_df = df[~df.index.isin(train_df.index)].sample(n=5, random_state=99).reset_index(drop=True)

finetuned_summaries = []
for article in tqdm(test_df['article'], desc="Fine-tuned evaluation"):
    summary = finetuned_summarizer(
        "summarize: " + article,
        max_length=130,
        min_length=30,
        do_sample=False
    )
    finetuned_summaries.append(summary[0]['summary_text'])

test_df['finetuned_summary'] = finetuned_summaries

# Calculate fine-tuned ROUGE scores
finetuned_results = rouge.compute(
    predictions=finetuned_summaries,
    references=test_df['summary'].tolist()
)

# Display comparison
print("\n" + "="*60)
print("MODEL PERFORMANCE COMPARISON")
print("="*60)
print(f"{'Metric':<12} {'Baseline':<12} {'Fine-tuned':<12} {'Δ':<12}")
print("-"*60)
print(f"{'ROUGE-1':<12} {baseline_results['rouge1']:<12.4f} {finetuned_results['rouge1']:<12.4f} {finetuned_results['rouge1']-baseline_results['rouge1']:+.4f}")
print(f"{'ROUGE-2':<12} {baseline_results['rouge2']:<12.4f} {finetuned_results['rouge2']:<12.4f} {finetuned_results['rouge2']-baseline_results['rouge2']:+.4f}")
print(f"{'ROUGE-L':<12} {baseline_results['rougeL']:<12.4f} {finetuned_results['rougeL']:<12.4f} {finetuned_results['rougeL']-baseline_results['rougeL']:+.4f}")
print("="*60)

# Show example
print("\nEXAMPLE OUTPUT:")
print("="*80)
print(f"Reference: {test_df['summary'].iloc[0][:150]}...")
print(f"\nGenerated: {test_df['finetuned_summary'].iloc[0]}")

Device set to use cuda:0


✓ Fine-tuned model loaded


Fine-tuned evaluation: 100%|██████████| 5/5 [00:07<00:00,  1.51s/it]



MODEL PERFORMANCE COMPARISON
Metric       Baseline     Fine-tuned   Δ           
------------------------------------------------------------
ROUGE-1      0.2944       0.4550       +0.1606
ROUGE-2      0.1824       0.3121       +0.1297
ROUGE-L      0.1992       0.3816       +0.1824

EXAMPLE OUTPUT:
Reference: Greg Rusedski was forced to withdraw from the Open 13 in Marseille on Thursday with a rib injury.But Rusedski was unable to take to the court because ...

Generated: the British number two had been scheduled to play qualifier Sebastien de Chaunac, who beat world number five Guillermo Coria 6-4 7-5 in round one. But Rusedski was unable to take to the court because of a problem with the left-hand side of his rib-cage. But third seed Joachim Johansson made it through after beating Frenchman Gilles Simon 7-6 6-3 while in the first match of the day, sixth seed Feliciano Lopez defeated Ivo Karlovic.
